# Practice 1 — Interactive Personal Budget Manager (Release 1)

**Business question:** Can we deliver the first integrated, validated budget manager release?

The notebook follows **Think → Predict → Test** for each behaviour:
1. **Think:** which business rule is this?
2. **Predict:** write the expected output *before* running anything.
3. **Test:** run the code and `assert` the prediction, so "it ran" becomes "it's verified".

The code lives in `src/budget_manager/`. This notebook imports it rather than re-implementing it,
so what's tested here is exactly what ships.

## 0. Setup

In [1]:
import sys, tempfile, subprocess
from pathlib import Path

# Find the project root (the folder that contains src/budget_manager), wherever the notebook is opened from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "budget_manager").exists())
sys.path.insert(0, str(ROOT / "src"))

from budget_manager.models import Transaction, KINDS, CATEGORIES_BY_KIND, CURRENCIES, FIELDS
from budget_manager.validation import parse_transaction, ValidationError
from budget_manager.storage import CsvStorage, read_csv, write_csv
from budget_manager.services import BudgetService, ReadOnlyError
from budget_manager.cli import format_table, format_report

WORK = Path(tempfile.mkdtemp())   # scratch folder, so the demo never touches real data
SAMPLE = ROOT / "data" / "sample_transactions.csv"
print("Project root:", ROOT.name)

Project root: budget-manager


## 1. Requirements as acceptance criteria

Each rule has a concrete input and an expected, observable result.

| # | Requirement | Example input | Expected result |
|---|---|---|---|
| R1 | Amount must be > 0 | `-12.00`, `0` | Rejected: *amount must be > 0* |
| R2 | Max 2 decimals, dot separator | `64.999`, `18,50` | Rejected, never rounded |
| R3 | Date is a real ISO date | `04/09/2026`, `2026-02-30` | Rejected |
| R4 | `kind` ∈ {expense, income} | `refund` | Rejected |
| R5 | Unknown category policy = **reject + list allowed values** | `gadgets` | Rejected |
| R6 | Currency ∈ {EUR, USD, GBP} | `JPY` | Rejected |
| R7 | IDs unique, format `T001` | `T013` twice | Second row rejected |
| R8 | File errors are located by row | bad CSV row | `Row N: <problem>` |
| R9 | Saved data survives a restart | add → reload | Same transactions |
| R10 | Undo is persisted, even after a crash | add, add, crash, undo | File holds only the first |

## 2. Data design: the transaction schema

In [2]:
print("Fields   :", FIELDS)
print("Kinds    :", KINDS)
print("Currency :", CURRENCIES)
for kind, cats in CATEGORIES_BY_KIND.items():
    print(f"{kind:<8} : {', '.join(cats)}")

tx = parse_transaction({"id": "T001", "date": "2026-09-04", "kind": "expense",
                        "category": "food", "amount": "18.50", "currency": "EUR"})
tx

Fields   : ('id', 'date', 'kind', 'category', 'amount', 'currency')
Kinds    : ('expense', 'income')
Currency : ('EUR', 'USD', 'GBP')
expense  : food, housing, transport, utilities, health, entertainment, education, other
income   : salary, freelance, gift, other


Transaction(id='T001', date=datetime.date(2026, 9, 4), kind='expense', category='food', amount=18.5, currency='EUR')

## 3. Normal case: add → list → report

**Think:** the core journey has to work end to end.
**Predict:** with a salary of 2500.00 and food expenses of 18.50 and 6.50, expenses should total **25.00** and the balance should be **2475.00**.

In [3]:
svc = BudgetService(CsvStorage(WORK / "normal.csv"))
svc.add({"date": "2026-09-01", "kind": "income",  "category": "salary", "amount": "2500.00", "currency": "EUR"})
svc.add({"date": "2026-09-04", "kind": "expense", "category": "food",   "amount": "18.50",   "currency": "EUR"})
svc.add({"date": "2026-09-05", "kind": "expense", "category": "food",   "amount": "6.50",    "currency": "EUR"})

print(format_table(svc.all()), end="\n\n")
print(format_report(svc, svc.all()))

eur = svc.summarize()["EUR"]
assert (eur.income, eur.expense, eur.balance) == (2500.00, 25.00, 2475.00)
assert [t.id for t in svc.all()] == ["T001", "T002", "T003"]
print("\n✅ prediction confirmed")

ID     DATE       KIND     CATEGORY            AMOUNT CUR
---------------------------------------------------------
T001   2026-09-01 income   salary         +   2500.00 EUR
T002   2026-09-04 expense  food           -     18.50 EUR
T003   2026-09-05 expense  food           -      6.50 EUR
3 transaction(s)

=== EUR (3 transaction(s)) ===
  Income :      2500.00
  Expense:        25.00
  Balance:      2475.00
  Expenses by category:
    food               25.00  (100.0%)

✅ prediction confirmed


## 4. Boundary cases: amount (R1, R2)

**Predict:** `0.01` and `1000000.00` are accepted (they're the smallest and largest valid values). `0`, `-12.00`,
`1000000.01` and `18.505` are rejected, each with a message saying how to fix it.

In [4]:
def try_amount(value):
    try:
        parse_transaction({"id": "T001", "date": "2026-09-04", "kind": "expense",
                           "category": "food", "amount": value, "currency": "EUR"})
        return "ACCEPTED"
    except ValidationError as e:
        return "REJECTED: " + e.errors[0]

predicted = {"0.01": True, "1000000.00": True, "0": False, "-12.00": False,
             "1000000.01": False, "18.505": False, "18,50": False}
for value, should_pass in predicted.items():
    result = try_amount(value)
    print(f"{value:>11} -> {result}")
    assert result.startswith("ACCEPTED") == should_pass
print("\n✅ all amount boundaries behave as predicted")

       0.01 -> ACCEPTED
 1000000.00 -> ACCEPTED
          0 -> REJECTED: amount must be > 0
     -12.00 -> REJECTED: amount must be > 0 (got -12.00); use kind=expense for money out
 1000000.01 -> REJECTED: amount must be <= 1,000,000.00 (got 1000000.01)
     18.505 -> REJECTED: amount '18.505' must have at most 2 decimal places
      18,50 -> REJECTED: amount '18,50' uses a comma; use a dot for decimals, e.g. 18.50

✅ all amount boundaries behave as predicted


## 5. Boundary cases: dates, kind, category, currency (R3–R6)

**Predict:** a leap day in a leap year (`2028-02-29`) is valid, while `2026-02-30` and `04/09/2026` are not.
An unknown category is **rejected** (not remapped to `other`), and the message lists the allowed values.
A row with several problems reports **all** of them at once.

In [5]:
assert parse_transaction({"id": "T001", "date": "2028-02-29", "kind": "expense",
                          "category": "food", "amount": "1", "currency": "EUR"}).date.day == 29

for bad in ({"date": "2026-02-30"}, {"date": "04/09/2026"}, {"kind": "refund"},
            {"category": "gadgets"}, {"kind": "income", "category": "food"}, {"currency": "JPY"}):
    row = {"id": "T001", "date": "2026-09-04", "kind": "expense",
           "category": "food", "amount": "10", "currency": "EUR", **bad}
    try:
        parse_transaction(row)
        raise AssertionError(f"{bad} should have been rejected")
    except ValidationError as e:
        print(f"{str(bad):<40} -> {e.errors[0]}")

try:
    parse_transaction({"id": "T001", "date": "2026-09-04", "kind": "refund",
                       "category": "food", "amount": "abc", "currency": "JPY"})
except ValidationError as e:
    assert len(e.errors) == 3
    print("\nMulti-error row reports", len(e.errors), "problems at once ✅")

{'date': '2026-02-30'}                   -> date '2026-02-30' is not a real calendar date
{'date': '04/09/2026'}                   -> date '04/09/2026' must use ISO format YYYY-MM-DD, e.g. 2026-09-04
{'kind': 'refund'}                       -> kind 'refund' must be one of: expense, income
{'category': 'gadgets'}                  -> category 'gadgets' is not allowed for expense; use one of: food, housing, transport, utilities, health, entertainment, education, other
{'kind': 'income', 'category': 'food'}   -> category 'food' is not allowed for income; use one of: salary, freelance, gift, other
{'currency': 'JPY'}                      -> currency 'JPY' must be one of: EUR, USD, GBP

Multi-error row reports 3 problems at once ✅


## 6. Malformed file: row-located errors (R7, R8)

**Predict:** `sample_transactions.csv` has 18 data rows. **7** of them are valid, and each invalid row is reported as
`Row N: ...`. The spreadsheet row numbers count the header as row 1, so the `-12.00` row is **Row 5**.

In [6]:
result = read_csv(SAMPLE)
print(f"Valid rows: {len(result.transactions)}\n")
for err in result.errors:
    print(err)

assert len(result.transactions) == 7
assert "Row 5: amount must be > 0 (got -12.00); use kind=expense for money out" in result.errors
assert "Row 15: id T013 duplicates row 14" in result.errors
print("\n✅ every bad row located and explained")

Valid rows: 7

Row 5: amount must be > 0 (got -12.00); use kind=expense for money out
Row 7: category 'gadgets' is not allowed for expense; use one of: food, housing, transport, utilities, health, entertainment, education, other
Row 8: amount '18,50' uses a comma; use a dot for decimals, e.g. 18.50
Row 9: date '04/09/2026' must use ISO format YYYY-MM-DD, e.g. 2026-09-04
Row 10: date '2026-02-30' is not a real calendar date
Row 11: kind 'refund' must be one of: expense, income
Row 12: amount '64.999' must have at most 2 decimal places
Row 13: currency 'JPY' must be one of: EUR, USD, GBP
Row 14: amount must be > 0
Row 15: id T013 duplicates row 14
Row 19: category is required

✅ every bad row located and explained


**Policy check:** a file with invalid rows has to become **read-only**. Saving would rewrite the file
without the bad rows, which would silently delete the user's data.
**Predict:** `add` raises `ReadOnlyError` and the file on disk stays byte-for-byte the same.

In [7]:
copy = WORK / "broken.csv"
copy.write_text(SAMPLE.read_text(encoding="utf-8"), encoding="utf-8")
before = copy.read_bytes()

svc_broken = BudgetService(CsvStorage(copy))
try:
    svc_broken.add({"date": "2026-09-20", "kind": "expense", "category": "food", "amount": "1", "currency": "EUR"})
except ReadOnlyError as e:
    print("Blocked:", e)

assert copy.read_bytes() == before
print("✅ file untouched")

Blocked: changes are blocked: the data file has 11 invalid row problem(s), and saving now would delete those rows. Fix them first (see load_errors / the warning above).
✅ file untouched


## 7. Persistence round-trip (R9)

**Predict:** if we add transactions, throw the service away and build a new one from the same file,
we get back identical `Transaction` objects.

In [8]:
path = WORK / "roundtrip.csv"
first = BudgetService(CsvStorage(path))
first.add({"date": "2026-09-04", "kind": "expense", "category": "food",  "amount": "18.50", "currency": "EUR"})
first.add({"date": "2026-09-14", "kind": "income",  "category": "freelance", "amount": "400", "currency": "USD"})

second = BudgetService(CsvStorage(path))
print(path.read_text())
assert second.all() == first.all()
assert second.load_errors == []
print("✅ save → load gives identical data")

id,date,kind,category,amount,currency
T001,2026-09-04,expense,food,18.50,EUR
T002,2026-09-14,income,freelance,400.00,USD

✅ save → load gives identical data


## 8. Stretch: undo without breaking persistence (R10)

**Think:** undo can't just delete from memory. The file has to reflect the true state, even after a crash.
**Design:** "last transaction" is defined as the **last row in the file**, and every change is written to disk *before*
memory is updated (atomic temp-file + `os.replace`).
**Predict:** add T001 and T002, "crash", reload, then undo. That should remove T002, and the file should then contain only T001.

In [9]:
path = WORK / "undo.csv"
s = BudgetService(CsvStorage(path))
s.add({"date": "2026-09-04", "kind": "expense", "category": "food", "amount": "1.00", "currency": "EUR"})
s.add({"date": "2026-09-05", "kind": "income",  "category": "gift", "amount": "9.00", "currency": "EUR"})
del s                                   # simulated crash: in-memory state is gone

after_crash = BudgetService(CsvStorage(path))
removed = after_crash.undo_last()
print("Undid:", removed.id)
print(path.read_text())

assert removed.id == "T002"
assert [t.id for t in read_csv(path).transactions] == ["T001"]
assert BudgetService(CsvStorage(path)).undo_last().id == "T001"   # can keep undoing
assert BudgetService(CsvStorage(path)).undo_last() is None         # empty -> nothing to undo
print("✅ undo is persisted and crash-safe")

Undid: T002
id,date,kind,category,amount,currency
T001,2026-09-04,expense,food,1.00,EUR

✅ undo is persisted and crash-safe


## 9. Full automated test suite

The checks in this notebook are also in `tests/test_budget_manager.py`, so they can be run without the notebook.

In [10]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", str(ROOT / "tests")],
                   cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-400:] or r.stderr[-400:])
assert r.returncode == 0

.................................                                        [100%]
33 passed in 0.09s



## 10. Design rationale

The program is split into layers (**CLI → service → validation/models → storage**), and dependencies point toward the business
rules. `parse_transaction` is the only way raw input becomes a `Transaction`, so interactive answers, command-line flags and CSV rows
all follow the same rules. The key business decision: a data file with invalid rows opens **read-only** rather than being
"cleaned", because dropping or fixing rows silently would make the report look right while hiding real money movements.

## 11. Exit ticket

- **🧪 What did I verify?** In section 8, after a simulated crash and reload, `undo_last()` removed `T002`, and the file on
  disk then held only `T001` (asserted, not just observed).
- **⚡ Which boundary changed my design?** The duplicate-ID check. At first it only compared *valid* rows, so a second `T013`
  slipped through when the first `T013` was itself invalid (amount `0`). The check now runs on the raw ID before validation (section 6).
- **🔧 What would I refactor next?** Switch `amount` from `float` to `decimal.Decimal`, and add an operation log to `BudgetService`
  so Release 2 can support multi-step undo and edit/delete by ID.

## 12. AI disclosure

See `AI_DISCLOSURE.md`. The code and this notebook were drafted with an AI assistant (Claude). I reviewed and ran every cell and test.